In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df=pd.read_csv('/kaggle/input/california-house-prices/train.csv')
df.info()
df.sample(3)

## Feature engineering
以下是一些平凑的代码。目标是做半自动特征工程，有些功能并未实现，可能有bug。就本项目来说还能用

In [ ]:
df_train=df.copy()
df_test=pd.read_csv('/kaggle/input/california-house-prices/test.csv')
for field in ['Listed On', 'Last Sold On']:
    df_train[field]=pd.to_datetime(df[field])
    df_test[field]=pd.to_datetime(df_test[field])

In [ ]:
cate_cols = []
num_cols = []
date_cols = []
dtypes = df_train.dtypes
for col, dtype in dtypes.items():
    if dtype=='object':
        cate_cols.append(col)
    elif dtype.name.startswith('datetime'):
        date_cols.append(col)
    else:
        num_cols.append(col)

In [ ]:
id_col = 'Id'
target_col = 'Sold Price'

for col in [id_col, target_col]:
    num_cols.remove(col)

print(cate_cols, num_cols, date_cols)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from pandas.api.types import is_string_dtype, is_numeric_dtype

class Num_Features(BaseEstimator, TransformerMixin):
    def __init__(self, cols = [], fillna = False, addna = False):
        self.fillna = fillna
        self.cols = cols
        self.addna = addna
        self.na_cols = []
        self.imputers = {}
    def fit(self, X, y=None):
        for col in self.cols:
            if self.fillna:
                self.imputers[col] = X[col].median()
            if self.addna and X[col].isnull().sum():
                self.na_cols.append(col)
        print(self.na_cols, self.imputers)
        return self
    def transform(self, X, y=None):
        df = X.loc[:, self.cols]
        for col in self.imputers:
            df[col].fillna(self.imputers[col], inplace=True)
        for col in self.na_cols:
            df[col+'_na'] = pd.isnull(df[col])
        return df

In [ ]:
class Imputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy, fill_value):
        self.strategy = strategy
        self.fill_value = fill_value
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        for col, content in X.items():
            X[col].fillna(self.fill_value, inplace=True)
        return X

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, LabelBinarizer, StandardScaler
from sklearn.impute import SimpleImputer
num_pipeline = Pipeline([
    ('select_num', Num_Features(cols=num_cols, fillna='median', addna=True)),
])
X_num = num_pipeline.fit_transform(df_train)

In [ ]:
class CatEncoder(BaseEstimator, TransformerMixin):
    def __init__(self,cols, max_n_cat=7, onehot_cols=[], orders={}):
        self.cols = cols
        self.onehot_cols=onehot_cols
        self.cats = {}
        self.max_n_cat = max_n_cat
        self.orders = orders
    def fit(self, X, y=None):
        df_cat =  X.loc[:, self.cols]
        for n,c in df_cat.items():
            df_cat[n].fillna('NAN', inplace=True)
            df_cat[n] = c.astype('category').cat.as_ordered()
            if n in self.orders:
                df_cat[n].cat.set_categories(self.orders[n], ordered=True, inplace=True)
            cats_count = len(df_cat[n].cat.categories)
            if cats_count<=2 or cats_count>self.max_n_cat:
                self.cats[n] = df_cat[n].cat.categories
                if n in self.onehot_cols:
                    self.onehot_cols.remove(n)
            elif n not in self.onehot_cols:
                self.onehot_cols.append(n)

        print(self.onehot_cols)
        return self
    def transform(self, df, y=None):
        X = df.loc[:, self.cols]
        for col in self.cats:
            X[col].fillna('NAN', inplace=True)
            X.loc[:,col] = pd.Categorical(X[col], categories=self.cats[col], ordered=True)
            X.loc[:,col] = X[col].cat.codes

#         for n,c in X.items():
#             if n in self.cats:
#                 X[n] = pd.Categorical(c, categories=self.cats[n], ordered=True)
#                 X[n] = X[n].cat.codes + 1
#             else:
#                 X[n] = c.astype('category').cat.as_ordered()
        if len(self.onehot_cols):
            df_1h = pd.get_dummies(X[self.onehot_cols], dummy_na=True)
            df_drop=X.drop(self.onehot_cols,axis=1)
            return pd.concat([df_drop, df_1h], axis=1)

        return X

In [ ]:
cat_pipeline = Pipeline([
    ('cat_encoder', CatEncoder(cols=cate_cols))
])
X_cate = cat_pipeline.fit_transform(df_train)

In [ ]:
def add_datepart(df, field_name, prefix=None, drop=True, time=False):
    field = df[field_name]
    if prefix is None:
        prefix = re.sub('[Dd]ate$', '', field_name)
    attr = ['Year', 'Month', 'Week', 'Day', 'Dayofweek', 'Dayofyear', 'Is_month_end', 'Is_month_start', 'Is_quarter_end', 'Is_quarter_start', 'Is_year_end', 'Is_year_start']
    if time: attr = attr + ['Hour', 'Minute', 'Second']
    # Pandas removed `dt.week` in v1.1.10
    week = field.dt.isocalendar().week.astype(field.dt.day.dtype) if hasattr(field.dt, 'isocalendar') else field.dt.week
    for n in attr: df[prefix + n] = getattr(field.dt, n.lower()) if n != 'Week' else week
    mask = ~field.isna()
    df[prefix + 'Elapsed'] = np.where(mask,field.values.astype(np.int64) // 10 ** 9,np.nan)
    if drop: df.drop(field_name, axis=1, inplace=True)
    return df

In [ ]:
import re
class Datepart(BaseEstimator, TransformerMixin):
    def __init__(self, cols, time=False):
        self.cols = cols
        self.time = time
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        df_dates = X.loc[:, self.cols]
        for col in self.cols:
            add_datepart(df_dates, col, time=False)
        return df_dates
    
date_pipeline = Pipeline([
    ('datepart', Datepart(cols=date_cols)),
    ('imputer', Imputer(strategy="constant", fill_value=-1)),
])

In [ ]:
X_date = date_pipeline.fit_transform(df_train)

In [ ]:
y_train = np.log(df_train[target_col])
X_train = pd.concat([X_num, X_cate,X_date], axis=1)
X_train.shape, y_train.shape

# Hyper Parameter Tuning
use grid search to produce a baseline model

In [ ]:
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(oob_score=True, random_state=3, n_jobs=-1)
params ={
    'n_estimators': [200],# [300, 400, 500, 600, 700],
    'min_samples_leaf': [2],# [1, 2, 3, 5, 10, 25],
    'max_features': [0.5],# [None, 0.5, 'sqrt', 'log2'],
    'max_depth': [13],# [5, 6, 7, 8, 10, 15, 20],
    'min_samples_split': [2]# [2, 3, 4]
}

best_score = 0
for g in ParameterGrid(params):
    model.set_params(**g)
    model.fit(X_train, y_train)
    if model.oob_score_ > best_score:
        best_score = model.oob_score_
        best_grid = g
        print('oob:', best_score, best_grid)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
m = RandomForestRegressor(n_jobs=-1, n_estimators=200, oob_score=True, max_depth=17, min_samples_leaf=4, min_samples_split=2, max_features=0.5)
m.fit(X_train, y_train)
m.oob_score_

# Feature Selection
feature importance

In [ ]:
def rf_feat_importance(m, df):
    return pd.DataFrame({'cols':df.columns, 'imp':m.feature_importances_}).sort_values('imp', ascending=False)

fi = rf_feat_importance(m, X_train)
fi[:50]

In [ ]:
del_cols = []#'Annual tax amount', 'Bathrooms', 'Last Sold OnYear', 'Summary', 'Address', 'Last Sold OnDayofyear','City','Heating features',
            #'Last Sold OnDay', 'Listed OnDayofyear', 'Elementary School', 'Listed OnDay', 'Type', 'Full bathrooms',
           #'High School', 'Middle School', 'Parking']#, 'Listed OnYear' # 'Last Sold OnYear', 'Region', # [ ]
keep_cols = ['Listed Price', 'Tax assessed value', 'Annual tax amount', 'Last Sold Price', 'Total interior livable area', 'Zip']

Threshold = 0.0009
to_keep = fi[fi.imp>Threshold].cols
to_keep = [col for _, col in to_keep.items()]

In [ ]:
for col in del_cols:
    if col in to_keep:
        to_keep.remove(col)
for col in keep_cols:
    if col not in to_keep:
        to_keep.append(col)
print(to_keep)
df_keep = X_train[to_keep].copy()

In [ ]:
m1 = RandomForestRegressor(n_jobs=-1, random_state=3, n_estimators=300, oob_score=True, max_depth=13, min_samples_leaf=2, min_samples_split=2, max_features=0.5)
m1.fit(df_keep, y_train)
print(m1.oob_score_)

In [ ]:
m1.oob_score_

最朴素的特征选择方法，找出可以去除的特征，反复做不断提高oob_score

In [ ]:
cols = to_keep
scores = []
feats = []
for col in cols:
    tmp = to_keep.copy()
    if col in keep_cols:
        continue
    tmp.remove(col)
    df_tmp = X_train[tmp].copy()
    m1 = RandomForestRegressor(n_jobs=-1, random_state=3, n_estimators=30, oob_score=True, max_depth=13, min_samples_leaf=2, min_samples_split=2, max_features=0.5)
    m1.fit(df_tmp, y_train)
    scores.append(m1.oob_score_)
    feats.append(col)
#     print(col, m1.oob_score_)

to_del = sorted(zip(scores, feats), reverse=True)
to_del

In [ ]:
# 最好提交的特征，18个
to_keep_final=['Listed Price', 'Tax assessed value', 'Last Sold Price', 'Zip', 'Total interior livable area', 'Elementary School Score', 
'Listed OnElapsed', 'Last Sold OnElapsed',
'Full bathrooms', 'Year built', 
'Listed OnYear', 
'Lot', 
'Parking','Type', 'Middle School Score', 'High School Distance', 'Elementary School Distance', 'Bedrooms']
# to_keep_final=['Listed Price', 'Tax assessed value', 'Last Sold Price', 'Zip', 'Total interior livable area', 'Listed OnElapsed', 'Elementary School Score', 'Last Sold OnElapsed', 'Year built', 'Listed OnYear', 'High School Distance', 'Lot', 'Parking', 'Middle School Score', 'Elementary School Distance', 'Region', 'Bedrooms', 'High School Score', 'Heating', 'Appliances included', 'Flooring', 'Middle School Distance']
X_train_final = X_train[to_keep_final].copy()

In [ ]:
# 2nd pass grid search to determine the final parameters
from sklearn.model_selection import ParameterGrid
model = RandomForestRegressor(oob_score=True, random_state=3, n_jobs=-1, max_features=0.5)
params ={
    'n_estimators': [500],# [400, 500, 600, 700, 900, 1000, 1100],
    'min_samples_leaf': [2],# [1, 2, 3, 5, 10, 25],
    'max_features': [0.5],# [0.5, 'sqrt', 'log2'],
    'max_depth': [10],# [5, 6, 7, 8],
    'min_samples_split': [2]# [2, 3, 4]
}

best_score = 0
for g in ParameterGrid(params):
    model.set_params(**g)
    model.fit(X_train_final, y_train)
    if model.oob_score_ > best_score:
        best_score = model.oob_score_
        best_grid = g
        print('best oob:', best_score, best_grid)

In [ ]:
# 最好成绩的超参数
model_final = RandomForestRegressor(n_jobs=-1, n_estimators=550, max_depth=17, min_samples_leaf=4, min_samples_split=2, max_features=0.45)
model_final.fit(X_train_final, y_train)

In [ ]:
X_test_num = num_pipeline.transform(df_test)
X_test_cate = cat_pipeline.transform(df_test)
X_test_date = date_pipeline.transform(df_test)
df_t = pd.concat([X_test_num, X_test_cate, X_test_date], axis=1)
df_t = df_t[to_keep_final]

In [ ]:
pred=model_final.predict(df_t)
df_pred=pd.DataFrame({'Id':df_test['Id'],'Sold Price': np.exp(pred)})
print(df_pred.head())
df_pred.to_csv('submission.csv', index=False)